# MACE Training Tutorial I

## Introduction

This tutorial introduces the MACE interatomic potential model [paper](https://arxiv.org/abs/2206.07697). MACE is an equivariant graph neural network that combines message passing with an Atomic Cluster Expansion (ACE)-based representation. The architecture has demonstrated strong performance across a range of atomistic benchmarks.

In this tutorial, we apply MACE to a metal-ligand system: thiolate-protected gold nanoclusters.

We first examine an available Au-S-C-H dataset and its reference labels. We then set up a MACE model for training and evaluation. Next, we use the trained model to perform molecular dynamics simulations. Finally, we analyze the model predictions and simulated structures using metrics relevant to a selected research question.


## Learning Objectives

By the end of this tutorial, you should be able to:

1. **Inspect an atomistic dataset and identify its configurations and reference labels.**
2. **Set up the main MACE parameters that control the model architecture and training procedure.**
3. **Train and evaluate a MACE model.**
4. **Perform molecular dynamics simulations using a trained MACE model.**
5. **Identify how MLIP models can be applied to a research question of interest.**

Google Colab provides limited computational resources, and GPU access depends on current availability.

From the menu bar, select `Runtime` > `Change runtime type`, and choose a GPU under `Hardware accelerator`. If a GPU is unavailable, switch to the CPU runtime.

Set the MACE device according to the available hardware: use `device="cuda"` for a GPU or `device="cpu"` for a CPU.

Throughout this tutorial, set the device according to the available hardware: use `device="cuda"` for a GPU or `device="cpu"` for a CPU.

## **0. Set up**

### Step 1: Install the Required Packages

In [ ]:
!pip install mace-torch nglview ipywidgets x3dase
!pip install numpy
!pip install ase
!pip install tblite

### Step 2: Set Up the Tutorial Repository and Working Directory

In [ ]:
!pip install "git+https://github.com/LeVHoangduc/mlipTools.git@develop"
!git clone -b develop https://github.com/LeVHoangduc/MLIP.git

%cd /content/MLIP/Tutorials

In [ ]:
!pwd

## **1. Dataset**

### 1.1. Dataset Construction

The dataset used in this tutorial is a subset of the published [Au-S-C-H dataset](https://materialsproject-contribs.s3.amazonaws.com/index.html#ausch_potential/) developed for an Atomic Cluster Expansion potential for thiolate-protected gold nanoclusters. The original fitting set was constructed mainly from simulations of the Au38Q and Au38T isomers and was augmented through active learning.

The reference energies and atomic forces in the original dataset were computed with density functional theory (DFT) using GPAW and the PBE exchange-correlation functional, following the settings reported in the [source study](https://doi.org/10.1021/acsnano.4c03094).

**In this tutorial**, we relabel the structures using a lower-cost semiempirical quantum-mechanical method. The raw structures therefore contain only the atomic numbers, $Z$, and Cartesian positions, $R$.

The dataset is stored in the extended XYZ (`extxyz`) format and can be read with the [Atomic Simulation Environment (ASE)](https://ase-lib.org/index.html), as shown below:

In [ ]:
from ase.io import read

train_db = read("dts/training_Au38.xyz",":")
test_db = read("dts/testing_Au38.xyz",":")

print("Number of configs in training set: ", len(train_db))
print("Number of configs in testing set: ", len(test_db))

print("Information stored in config.info in training set: \n", train_db[10].info)
print("Information stored in config.arrays in training set: \n", train_db[10].arrays.keys())

print("Information stored in config.info in testing set: \n", test_db[10].info)
print("Information stored in config.arrays in testing set: \n", test_db[10].arrays.keys())

**Let's visualize** one configuration

In [ ]:
from ase.io import read
from ase.visualize import view

atoms = read("dts/testing_Au38.xyz", index=10)

view(atoms, viewer="x3d")

### 1.2. Data Labeling

Given the computational resources available in this tutorial, we generate reference labels with a lower-cost semiempirical tight-binding method.

Using the `tblite` [package](https://tblite.readthedocs.io/en/latest/), GFN1-xTB is applied to compute the reference energies and forces. The MACE model trained in this tutorial will therefore approximate the GFN1-xTB potential-energy surface.

An **isolated-atom energy** is the energy of a single atom placed in vacuum, i.e., with no bonds, nearby atoms, or interactions.

ML interatomic potentials, such as MACE, are usually trained on **atomization energies**, defined here as:

$$
E^{\mathrm{atm}} = E^{\mathrm{tot}} - \sum_i E_{Z_i}^{0},
$$

where:

* $E^{\mathrm{tot}}$: total energy of the system;
* $E_{Z_i}^{0}$: isolated-atom energy of the element corresponding to atom $i$ (e.g., H or Au).

This expression simply subtracts the isolated-atom energies, $E^0$, from the total energy of the system.

The total energy is then predicted from the atomic environments.


**Isolated energy** is the energy of a single atom placed in vacuum, i.e with no bonds, no nearby atoms, and no interactions.

ML interatomic potentials (like MACE) usually train on **atomization energies**, defined as:

$$E^{\text{atm}} = E^{\text{tot}} - \sum_i E_{Z_i}^{0}$$

where:

*   $E^{\text{tot}}$: total energy of the molecule
*   $E^{0}_i$: energy of the corresponding isolated atom type (e.g., one H, one Au)

It simply subtract the isolated eneriges $E^{0}$ from the total energy of the system.

Then, the total energy will be predicted from atomic environments






**Set up the isolated-atom reference energies for the Au-S-C-H system**

All configurations contain four chemical elements: Au, S, C, and H. We compute one isolated-atom reference energy for each element and define

$$
E^{\mathrm{ref}} = E^{\mathrm{tot}}
-N_{\mathrm{Au}}E_{\mathrm{Au}}^0
-N_{\mathrm{S}}E_{\mathrm{S}}^0
-N_{\mathrm{C}}E_{\mathrm{C}}^0
-N_{\mathrm{H}}E_{\mathrm{H}}^0.
$$

We prepend the four isolated-atom configurations to the dataset and assign `config_type=IsolatedAtom` to each of them.


In [ ]:
from ase import Atoms
from ase.io import read,write

db = read('dts/training_Au38.xyz', ':')

db = [Atoms('Au'), Atoms('S'), Atoms('C'), Atoms('H')] + db  # add isolated atoms

for at in db[:4]:
    at.info['config_type'] = 'IsolatedAtom'

print("Number of configs in database: ", len(db))

In [ ]:
for at in db[:4]:
  print(at.info)

### Brief Introduction to GFN1-xTB

GFN1-xTB is a semiempirical extended tight-binding method designed for efficient calculations of molecular geometries, energies, and related properties across broad chemical space. Its parameterization covers the elements required here, including H, C, S, and Au. In this tutorial, GFN1-xTB provides the energy and force labels at a substantially lower computational cost than DFT.

**Run GFN1-xTB calculations** for the first 15 entries: four isolated atoms and one Au38 configuration.

In [ ]:
from tqdm import tqdm
from tblite.ase import TBLite

xtb_calc = TBLite(method="GFN1-xTB")

for at in tqdm(db[:15]):
    at.calc = xtb_calc
    at.info['energy_gfn1'] = at.get_potential_energy()
    at.arrays['forces_gfn1'] = at.get_forces()

# write('dts/training_Au38_xtb.xyz', db) # Uncomment this line to write labeled data

In [ ]:
from ase_atom_tools import aseTools as at

print(f"E0s:\n{at.get_E0s(db, 'energy_gfn1')}")
print(f"Information stored in config.info:\n{db[4].info}")
print(f"Information stored in config.arrays:\n{db[4].arrays.keys()}")

After labeling, each configuration stores the GFN1-xTB total energy under the `energy_gfn1` key and the atomic forces under the `forces_gfn1` key.

Each Au38(SCH3)24 configuration contains 158 atoms. Labeling the complete training and test sets can therefore take several hours.

> In one benchmark using 40 CPU cores, four threads per task, and ten parallel tasks, labeling 6,004 configurations took approximately `3 h 46 min 19 s`. This value is a hardware- and implementation-dependent runtime rather than a general performance estimate.

The complete datasets have therefore been precomputed. Let us inspect the files in the `dts` directory.


**Let's inspect** the isolated-atom energies, $E^0$, and the corresponding atomization energies, $E^{\mathrm{atm}}$, in the precomputed dataset.

In [ ]:
db = read("dts/training_Au38_xtb.xyz",":")
E0s = at.get_E0s(db,"energy_gfn1")

print(f"E0s:{E0s}")

In [ ]:
atoms = read("dts/training_Au38_xtb.xyz",index=10)

print(f"Total Energy: {atoms.info['energy_gfn1']}")
print(f"Atomization Energy: {at.get_atomization_energy(atoms,E0s,'energy_gfn1',peratom=False)}")
print(f"Atomization Energy per atom: {at.get_atomization_energy(atoms,E0s,'energy_gfn1',peratom=True)}")

**In summary,** we generated GFN1-xTB energy and force labels for the Au-S-C-H training and test data.

The prepared dataset contains isolated-atom reference energies, total energies, and atomic forces.

The data are now ready for MACE training and evaluation.


## **2. MACE Potential Training**

## Configuration File Structure

The MACE configuration file is organized into five main sections:

1. **General model settings** — basic model, output, device, and reproducibility settings.
2. **Model architecture** — defines the MACE network architecture and interaction range.
3. **Data configuration** — specifies the training, validation, and test datasets and target properties.
4. **Training settings** — controls the training procedure, stopping criteria, and model averaging strategies.
5. **Optimizer and scheduler settings** — controls parameter optimization and learning-rate scheduling.


### 1. General Model Settings

This section defines the general information of the training run, including the model type, output locations, computational device, and reproducibility settings.

- `model`: Specifies the type of model used for training. In this case, the standard MACE architecture is selected.
- `name`: Defines the name of the training run and is commonly used to identify the resulting model.
- `model_dir`: Directory where the trained model is stored.
- `log_dir`: Directory where training logs are written.
- `checkpoints_dir`: Directory used to save intermediate checkpoints during training.
- `results_dir`: Directory used to store training and evaluation results.
- `device`: Specifies the computational device used for training. Using `cuda` enables GPU acceleration.
- `seed`: Random seed used to make the training procedure more reproducible.


### 2. Model Architecture

This section defines the structure and complexity of the MACE neural network.

- `hidden_irreps`: Defines the hidden equivariant feature representations used by the network. For example, `128x0e + 128x1o` corresponds to 128 even scalar channels and 128 odd vector channels.
- `num_interactions`: Specifies the number of interaction blocks used for message passing between neighboring atoms.

- `correlation`: Sets the maximum body order of the basis functions built in each layer. With correlation = $\nu$, each basis function couples the central atom $i$ with up to $\nu$ neighbours, giving ($\nu$+1)-body terms; e.g. `correlation=3` gives up to 4-body (ijkl) terms.
- `max_ell`: Maximum degree $\ell$ of the spherical harmonics $Y_\ell^m$ used to expand the direction to each neighbour; sets the model's angular resolution (default 3).

- `r_max`: Defines the cutoff radius, in Å., used to determine which neighboring atoms contribute to the local atomic environment.


### 3. Data Configuration

This section specifies the datasets used during training and the target properties that MACE should learn.

- `train_file`: Path to the dataset used for model training.
- `valid_fraction`: Fraction of the training dataset reserved for validation.
- `test_file`: Path to an independent dataset used to evaluate the trained model.
- `energy_key`: Name of the configuration-level energy property stored in the dataset.
- `forces_key`: Name of the atomic force array stored in the dataset.
- `E0s`: Defines the isolated-atom reference energies. MACE can read explicitly labeled `IsolatedAtom` configurations from the training file. If such references are unavailable, `E0s: average` estimates element-wise offsets by least-squares regression.


### 4. Training Settings

This section controls how the training process is performed, including batch processing, training duration, stopping criteria, and model averaging.

- `batch_size`: Number of atomic configurations processed together in each training step.
- `max_num_epochs`: Maximum number of complete passes through the training dataset.
- `patience`: Number of epochs without sufficient improvement before early stopping is triggered.
- `swa`: Enables Stochastic Weight Averaging, which averages model parameters during the later stages of training and may improve generalization.
- `ema`: Enables Exponential Moving Average of the model parameters during training.
- `ema_decay`: Decay coefficient controlling how strongly previous model parameters contribute to the exponential moving average.
- `restart_latest`: Allows training to resume from the most recent available checkpoint instead of starting from the beginning.


### 5. Optimizer and Scheduler Settings

This section defines how the neural-network parameters are updated and how the learning rate is adjusted during training.

- `lr`: Initial learning rate used by the optimizer.
- `weight_decay`: Regularization term applied to model parameters to help reduce overfitting and excessively large weights.
- `amsgrad`: Enables the AMSGrad variant of the Adam optimizer, which modifies the adaptive learning-rate update for improved numerical stability in some training problems.
- `clip_grad`: Sets the maximum gradient norm used for gradient clipping, helping prevent excessively large parameter updates.
- `scheduler`: Specifies the learning-rate scheduling strategy used during training.
- `lr_factor`: Multiplicative factor applied to the learning rate when the scheduler is activated.
- `scheduler_patience`: Number of epochs without improvement before the learning-rate scheduler reduces the learning rate.

### **2.1. Configuration File**

In [ ]:
%%writefile config/config-ausch.yml


# ============================================================
# 1. General Model Settings
# ============================================================

model: "MACE"
name: "mace_ausch_xtb"
model_dir: "models"
log_dir: "logs"
checkpoints_dir: "checkpoints"
results_dir: "logs"
device: "cuda"
seed: 123


# ============================================================
# 2. Model Architecture
# ============================================================

hidden_irreps: "128x0e + 128x1o"
num_interactions: 2
correlation: 3
max_ell: 3
r_max: 6.0

# ============================================================
# 3. Data Configuration
# ============================================================

train_file: "dts/training_Au38_xtb_100.xyz"
test_file: "dts/testing_Au38_xtb.xyz"
valid_fraction: 0.1
energy_key: "energy_gfn1"
forces_key: "forces_gfn1"


# ============================================================
# 4. Training Settings
# ============================================================

batch_size: 6
max_num_epochs: 2
patience: 200
swa: true
ema: true
ema_decay: 0.999
restart_latest: false

# ============================================================
# 5. Optimizer and Scheduler Settings
# ============================================================

lr: 0.01
weight_decay: 5e-7
amsgrad: true
clip_grad: 10.0
scheduler: "ReduceLROnPlateau"
lr_factor: 0.8
scheduler_patience: 10

### **2.2. Train the MACE Model**

In [ ]:
import warnings
warnings.filterwarnings("ignore")
from mace.cli.run_train import main as mace_run_train_main
import sys
import logging

def train_mace(config_file_path):
    logging.getLogger().handlers.clear()
    sys.argv = ["program", "--config", config_file_path]
    mace_run_train_main()

In [ ]:
train_mace("config/config-ausch.yml")

##### **Optional: Repeat the training with prepared 500, 1,500, and 5,000 training configurations.**

How does the test error change with training-set size?

> Remember to use a distinct model name for each training-set size.


### 2.3. Evaluate the MACE Model

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import os

os.makedirs("evals/mace_ausch_xtb/", exist_ok=True)

In [ ]:
from mace.cli.eval_configs import main as mace_eval_configs_main
import sys

def eval_mace(configs, model, output):
    sys.argv = ["program", "--configs", configs, "--model", model, "--output", output]
    mace_eval_configs_main()

##### **Let's practice:** Select your trained model for evaluation.

In [ ]:
# Implement your code here

#evaluate the training set
eval_mace(configs="dts/training_Au38_xtb_100.xyz",
          model="models/mace_ausch_stagetwo.model", # <-- Add the path to your trained model
          output="evals/mace_ausch_xtb/Au38_xtb_100_train.xyz")

#evaluate the test set
eval_mace(configs="dts/testing_Au38_xtb.xyz",
          model="models/mace_ausch_stagetwo.model", # <-- Add the path to your trained model
          output="evals/mace_ausch_xtb/Au38_xtb_test.xyz")

#### **Visualize the Evaluation Results**

In [ ]:
from mlip_utils import plot

eval_train_db = read("evals/mace_ausch_xtb/Au38_xtb_100_train.xyz",index=":")
eval_test_db = read("evals/mace_ausch_xtb/Au38_xtb_test.xyz",index=":")

labs = ["GFN1-xTB","MACE"]

energy_label = ["energy_gfn1","MACE_energy"]
forces_label = ["forces_gfn1","MACE_forces"]

unit = "meV"

plot.RMSE(eval_train_db,"Training Set",labs,energy_label,forces_label,unit)
plot.RMSE(eval_test_db,"Testing Set",labs,energy_label,forces_label,unit)

### 2.4. Training from Scratch and Using Foundation Models

This tutorial also provides a MACE model trained from scratch on 5,004 configurations. Because of the limited computational resources available in Colab, we trained the model in advance and provide it as `mace_ausch_run-123_stagetwo.model`. The model performance has improved significantly.

Moreover, several pretrained MACE foundation models are available as additional baselines in the `mace_foundation_models` folder.

##### **Let’s check them out.** Use the trained model and the foundation models for evaluation and compare the results.

##### **Evaluate the From-Scratch MACE Model**

In [ ]:
#evaluate the training set
eval_mace(configs="dts/training_Au38_xtb_100.xyz",
          model="models/mace_ausch_run-123_stagetwo.model",
          output="evals/mace_ausch_xtb/Au38_xtb_100_train.xyz")

#evaluate the test set
eval_mace(configs="dts/testing_Au38_xtb.xyz",
          model="models/mace_ausch_stagetwo.model",
          output="evals/mace_ausch_xtb/Au38_xtb_test.xyz")

##### **Evaluate the MACE Foundation Models**

##### **Let's practice:** Select one or more foundation models from `mace_foundation_models`, evaluate them on the same structures, and visualize the results.

> Treat these models as transferability baselines. Their training data and reference methods differ from GFN1-xTB, and gold-thiolate configurations may be outside their well-sampled domain.


In [ ]:
# Implement your code here

#evaluate the training set
eval_mace(configs="dts/training_Au38_xtb_100.xyz",
          model="evals/path_to_foundation_model", # <-- Choose the foundation model
          output="tests/mace01/Au38_xtb_100_train.xyz")

#evaluate the test set
eval_mace(configs="dts/testing_Au38_xtb.xyz",
          model="evals/path_to_foundation_model", # # <-- Choose the foundation model
          output="tests/mace01/Au38_xtb_test.xyz")

##### **Let's practice:** Visualize the results obtained with the from-scratch model and the foundation models.

We provide the evaluation files generated by the from-scratch MACE model trained on 5,004 configurations, as mentioned above.

You can directly `unzip` the files and visualize the results.

In [ ]:
!unzip -o evals/mace_ausch_scratch.zip -d evals

In [ ]:
# Implement your code here

# 1. Scratch model

from mlip_utils import plot

eval_train_db = read("evals/mace_ausch_scratch/scratch_train_eval.xyz",index=":")
# Exclude IsolatedAtom entries (E0s): they stretch the plot axes, obscure the 5,000 configurations,
# and do not require evaluation.
eval_train_db = [a for a in eval_train_db if a.info.get("config_type") != "IsolatedAtom"]

eval_test_db = read("evals/mace_ausch_scratch/scratch_test_eval.xyz",index=":")

labs = ["GFN1-xTB","MACE"]

energy_label = ["energy_gfn1","MACE_energy"]
forces_label = ["forces_gfn1","MACE_forces"]

unit = "eV" # eV, meV

plot.RMSE(eval_train_db,"Training Set",labs,energy_label,forces_label,unit)
plot.RMSE(eval_test_db,"Testing Set",labs,energy_label,forces_label,unit)

In [ ]:
# Implement your code here

# 2. Foundation models

from mlip_utils import plot

eval_train_db = read("evals/mace_ausch_scratch/path_to_train_eval_file.xyz",index=":") # <-- Add the evaluation path
eval_test_db = read("evals/mace_ausch_scratch/path_to_test_eval_file.xyz",index=":") # <-- Add the evaluation path

labs = ["GFN1-xTB","MACE"]

energy_label = ["energy_gfn1","MACE_energy"]
forces_label = ["forces_gfn1","MACE_forces"]

unit = "meV"

plot.RMSE(eval_train_db,"Training Set",labs,energy_label,forces_label,unit)
plot.RMSE(eval_test_db,"Testing Set",labs,energy_label,forces_label,unit)

## **3. Applications of the MACE Potential**

### 3.1. Predict *Unseen* Gold-Thiolate Structures


In this section, we use the previously trained MACE models to predict the properties of known $\mathrm{Au}_n(\mathrm{SCH}_3)_m$ nanoclusters.

The benchmark set contains 16 experimentally reported structures with cluster sizes ranging from Au18 to Au144. To maintain consistency with the Au-S-C-H model chemistry, the original ligands in the reported crystal structures were replaced with $\mathrm{SCH}_3$ groups following the protocol described in the [source paper](https://doi.org/10.1021/acsnano.4c03094).


##### **Let's practice:** Relax the structures using the [ASE optimization tools](https://ase-lib.org/ase/optimize.html) with each of the following calculators:

1. The MACE potentials evaluated in this tutorial

2. GFN1-xTB as the geometry-optimization reference


In [ ]:
# Implement your code here

# 1. Using MACE potentials

In [ ]:
# Implement your code here

# 2. Using xTB calculations

### 3.2. Molecular Dynamics (MD) simulation

Use the from-scratch MACE model to run a molecular dynamics simulation.

As a simple example, simulate an Au25 nanocluster in the NVT ensemble at 300 K.

The MACE trajectory can then be compared with a trajectory generated using GFN1-xTB under the same initial conditions and simulation settings.


In [ ]:
# ============================================================
# Simple NVT MD with MACE/xTB + ASE (single structure)
# System: Au_n(SCH3)_m
# Flow:
#   1. minimize()     : geometry optimization
#   2. MDsimulation() : velocities at T (fixed seed) + NVT run, save trajectory
#   Timestep: 1.0 fs
# ASE units: eV, Å; multiply times by units.fs
# ============================================================
import os
import time

import matplotlib.pyplot as plt
import numpy as np
from ase import units
from ase.io import read, write
from ase.md.nose_hoover_chain import NoseHooverChainNVT
from ase.md.velocitydistribution import (
    MaxwellBoltzmannDistribution,
    Stationary,
    ZeroRotation,
)
from ase.optimize import BFGS
from IPython import display
from mace.calculators import MACECalculator

# -----------------------------
# MD settings
# -----------------------------
seed = 12345

dt = 1.0 * units.fs      # timestep 0.001 ps
tdamp = 50.0 * units.fs  # = Tdamp 0.05 ps in fix nvt
dump_every = 10          # save 1 frame every 10 steps (= dump 10)
log_every = 1000         # = thermo 1000
live_plot = True         # live E/T plot in Jupyter


# ============================================================
# 1. Minimization
# ============================================================
def minimize(input_structure, calc, optimizer=BFGS, fmax=0.01, steps=10_000, logfile="-"):
    print("========== Minimization ==========", flush=True)

    atoms = input_structure.copy()
    atoms.calc = calc

    # fmax = max force on a single atom (eV/Å)
    optimizer(atoms, logfile=logfile).run(fmax=fmax, steps=steps)
    return atoms


# ============================================================
# 2. NVT MD
# ============================================================
def MDsimulation(input_structure, calc, output_path, traj_logs, temp, nsteps):
    """
    input_structure : ase.Atoms (e.g. the minimized structure)
    calc            : ASE calculator (e.g. MACECalculator)
    output_path     : trajectory file (extxyz, can be opened in OVITO)
    traj_logs       : MD log file (Time, Etot, Epot, Ekin, T every log_every steps)
    temp            : temperature (K)
    nsteps          : number of MD steps (dt = 1 fs -> 10,000 steps = 10 ps)
    """
    for path in (output_path, traj_logs):
        os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    if os.path.exists(output_path):
        os.remove(output_path)

    atoms = input_structure.copy()
    atoms.calc = calc

    # -----------------------------
    # Create velocities at T
    # -----------------------------
    print(f"========== Create velocities at {temp:.0f} K ==========", flush=True)

    rng = np.random.default_rng(seed)
    MaxwellBoltzmannDistribution(atoms, temperature_K=temp, force_temp=True, rng=rng)
    Stationary(atoms)
    ZeroRotation(atoms)

    # -----------------------------
    # NVT run
    # -----------------------------
    print(f"========== NVT at {temp:.0f} K: {nsteps} steps ==========", flush=True)

    dyn = NoseHooverChainNVT(atoms, timestep=dt, temperature_K=temp, tdamp=tdamp,
                             logfile=traj_logs, loginterval=log_every)

    def remove_drift():      # = fix momentum 100 linear 1 1 1 angular
        Stationary(atoms)
        ZeroRotation(atoms)

    dyn.attach(remove_drift, interval=100)

    # E/T vs time plot
    time_fs, energies, temperature = [], [], []
    fig, ax = plt.subplots(2, 1, figsize=(6, 6), sharex=True, gridspec_kw={"hspace": 0})
    line_e, = ax[0].plot([], [], color="b")
    line_t, = ax[1].plot([], [], color="r")
    ax[0].set_ylabel("E (eV/atom)")
    ax[1].set_ylabel("T (K)")
    ax[1].set_xlabel("Time (fs)")

    def update_plot():
        line_e.set_data(time_fs, energies)
        line_t.set_data(time_fs, temperature)
        for a in ax:
            a.relim()
            a.autoscale_view()

    def write_frame():
        write(output_path, atoms, format="extxyz", append=True,
              columns=["symbols", "positions", "momenta"])
        time_fs.append(dyn.get_time() / units.fs)
        energies.append(atoms.get_potential_energy() / len(atoms))
        temperature.append(atoms.get_temperature())

        if live_plot:
            update_plot()
            display.clear_output(wait=True)
            display.display(fig)

    dyn.attach(write_frame, interval=dump_every)

    t0 = time.time()
    dyn.run(nsteps)

    update_plot()
    fig.savefig(os.path.splitext(traj_logs)[0] + ".png", dpi=150, bbox_inches="tight")
    plt.close(fig)           # avoid a duplicate figure at the end of the notebook cell

    print(f"========== DONE in {(time.time() - t0) / 60:.2f} min ==========")
    return atoms


##### Run Molecular Dynamics with the From-Scratch MACE Model

In [ ]:
mace_model = "models/mace_ausch_run-123_stagetwo.model"
name = "au25(sch3)18"
structure_file = "dts/Au25.xyz"

temp = 300.0             # K
nsteps = 10_000         # 10 ps

outprefix = f"{name}_{temp:.0f}K_{seed}"

atoms = read(structure_file)
atoms.pbc = False        # = boundary f f f

calc = MACECalculator(model_paths=mace_model,
                      device="cpu", # cuda
                      default_dtype="float64")

atoms_min = minimize(atoms, calc, logfile=f"MDsimulations/md_logs/{outprefix}_min.log")

atoms_md = MDsimulation(atoms_min, calc,
                        output_path=f"MDsimulations/traj/{outprefix}_nvt_equil.xyz",
                        traj_logs=f"MDsimulations/md_logs/{outprefix}_nvt_equil.log",
                        temp=temp, nsteps=nsteps)

##### Run Molecular Dynamics with GFN1-xTB

In [ ]:
from tblite.ase import TBLite

name = "au25(sch3)18"
structure_file = "dts/Au25.xyz"

temp = 300.0             # K
nsteps = 10_000          # 10 ps

outprefix = f"{name}_xtb_{temp:.0f}K_{seed}"

atoms = read(structure_file)
atoms.pbc = False        # = boundary f f f

calc = TBLite(method="GFN1-xTB")

atoms_min = minimize(atoms, calc, logfile=f"MDsimulations/md_logs/{outprefix}_min_XTB.log")

atoms_md = MDsimulation(atoms_min, calc,
                        output_path=f"MDsimulations/traj/{outprefix}_XTB_nvt_equil.xyz",
                        traj_logs=f"MDsimulations/md_logs/{outprefix}_XTB_nvt_equil.log",
                        temp=temp, nsteps=nsteps)

### **Optional Practice 1: Analyze and Validate the Trajectories**

After applying the potential, validate its predictions against an appropriate reference method, such as DFT, whenever feasible.

It is also important to examine the bonding and structural properties predicted by the MLIP.

Each analysis metric describes a different aspect of the trajectory. Select one or more of the following analyses according to your interests:

1. The Au-S bond-length distribution

2. The root-mean-square displacement or aligned structural RMSD over the trajectory

3. The vibrational density of states obtained from the velocity autocorrelation function


In [ ]:
# Implement your code here

### **Optional Practice 2: Extend the Project**


This small project demonstrates how an efficient MLIP can reproduce a chosen reference potential at a substantially lower evaluation cost.

The tutorial takes you through a complete workflow for a thiolate-protected gold nanocluster: preparing reference data, training a MACE model, evaluating its errors, and running a short molecular dynamics simulation.

As a next step, train several candidate models, select the model using fixed validation and test metrics, and then perform longer simulations to investigate thermally activated structural changes.

The referenced [ACS Nano study](https://doi.org/10.1021/acsnano.4c03094) reported a 42 ns Au25 isomerization trajectory and a 134 ns two-cluster coalescence and equilibration trajectory. The latter showed the formation and desorption of $[\mathrm{AuSR}]_n$ fragments on an approximately 0.1 $\mu$s timescale.

**As an advanced practice,** investigate whether related events can be sampled with your model. Interpret any observed mechanism cautiously and verify that the trajectory remains within the model's validated domain.


# External sources

1. McCandler, C. A., Pihlajamäki, A., Malola, S., Häkkinen, H. & Persson, K. A.  
   **Gold–Thiolate Nanocluster Dynamics and Intercluster Reactions Enabled by a Machine Learned Interatomic Potential.**  
   *ACS Nano* **18**(29), 19014–19023 (2024).  
   https://doi.org/10.1021/acsnano.4c03094

2. Pihlajamäki, A., Hämäläinen, J., Linja, J., Nieminen, P., Malola, S., Kärkkäinen, T. & Häkkinen, H.  
   **Monte Carlo Simulations of Au38(SCH3)24 Nanocluster Using Distance-Based Machine Learning Methods.**  
   *The Journal of Physical Chemistry A* **124**(23), 4827–4836 (2020).  
   https://doi.org/10.1021/acs.jpca.0c01512

3. Sabooni Asre Hazer, M., Malola, S. & Häkkinen, H.  
   **Thermal Dynamics and Coalescence of Au144(SR)60 Clusters from a Machine-Learned Potential.**  
   *Nature Communications* **17**, 971 (2026).  
   https://doi.org/10.1038/s41467-025-67700-w

4. Batatia, I., Benner, P., Chiang, Y., Elena, A. M., Kovács, D. P., Riebesell, J. *et al.*  
   **A Foundation Model for Atomistic Materials Chemistry.**  
   *The Journal of Chemical Physics* **163**(18), 184110 (2025).  
   https://doi.org/10.1063/5.0297006

5. Batatia, I., Kovács, D. P., Simm, G. N. C., Ortner, C. & Csányi, G.  
   **MACE: Higher Order Equivariant Message Passing Neural Networks for Fast and Accurate Force Fields.**  
   *Advances in Neural Information Processing Systems* **35**, 11423–11436 (2022).

6. Chen, S., Wang, S., Zhong, J., Song, Y., Zhang, J., Sheng, H., Pei, Y. & Zhu, M.  
   **The Structure and Optical Properties of the [Au18(SR)14] Nanocluster.**  
   *Angewandte Chemie International Edition* **54**, 3145–3149 (2015).  
   https://doi.org/10.1002/anie.201410295

7. Das, A., Liu, C., Byun, H. Y., Nobusada, K., Zhao, S., Rosi, N. L. & Jin, R.  
   **Structure Determination of [Au18(SR)14].**  
   *Angewandte Chemie International Edition* **54**, 3140–3144 (2015).  
   https://doi.org/10.1002/anie.201410161

8. Yang, S., Chai, J., Song, Y., Fan, J., Chen, T., Wang, S., Yu, H., Li, X. & Zhu, M.  
   **In Situ Two-Phase Ligand Exchange: A New Method for the Synthesis of Alloy Nanoclusters with Precise Atomic Structures.**  
   *Journal of the American Chemical Society* **139**, 5668–5671 (2017).  
   https://doi.org/10.1021/jacs.7b00668

9. Das, A., Li, T., Li, G., Nobusada, K., Zeng, C., Rosi, N. L. & Jin, R.  
   **Crystal Structure and Electronic Properties of a Thiolate-Protected Au24 Nanocluster.**  
   *Nanoscale* **6**, 6458–6462 (2014).  
   https://doi.org/10.1039/C4NR01350F

10. Zhu, M., Eckenhoff, W. T., Pintauer, T. & Jin, R.  
    **Conversion of Anionic [Au25(SCH2CH2Ph)18]− Cluster to Charge Neutral Cluster via Air Oxidation.**  
    *The Journal of Physical Chemistry C* **112**, 14221–14224 (2008).  
    https://doi.org/10.1021/jp805786p

11. Higaki, T., Liu, C., Chen, Y., Zhao, S., Zeng, C., Jin, R., Wang, S., Rosi, N. L. & Jin, R.  
    **Oxidation-Induced Transformation of Eight-Electron Gold Nanoclusters: [Au23(SR)16]− to [Au28(SR)20]0.**  
    *The Journal of Physical Chemistry Letters* **8**, 866–870 (2017).  
    https://doi.org/10.1021/acs.jpclett.6b03061

12. Nimmala, P. R., Knoppe, S., Jupally, V. R., Delcamp, J. H., Aikens, C. M. & Dass, A.  
    **Au36(SPh)24 Nanomolecules: X-ray Crystal Structure, Optical Spectroscopy, Electrochemistry, and Theoretical Analysis.**  
    *The Journal of Physical Chemistry B* **118**, 14157–14167 (2014).  
    https://doi.org/10.1021/jp506508x

13. Qian, H., Eckenhoff, W. T., Zhu, Y., Pintauer, T. & Jin, R.  
    **Total Structure Determination of Thiolate-Protected Au38 Nanoparticles.**  
    *Journal of the American Chemical Society* **132**, 8280–8281 (2010).  
    https://doi.org/10.1021/ja103592z

14. Tian, S., Li, Y.-Z., Li, M.-B., Yuan, J., Yang, J., Wu, Z. & Jin, R.  
    **Structural Isomerism in Gold Nanoparticles Revealed by X-ray Crystallography.**  
    *Nature Communications* **6**, 8667 (2015).  
    https://doi.org/10.1038/ncomms9667

15. Zeng, C., Chen, Y., Liu, C., Nobusada, K., Rosi, N. L. & Jin, R.  
    **Gold Tetrahedra Coil Up: Kekulé-Like and Double Helical Superstructures.**  
    *Science Advances* **1**, e1500425 (2015).  
    https://doi.org/10.1126/sciadv.1500425

16. Zhuang, S., Liao, L., Zhao, Y., Yuan, J., Yao, C., Liu, X., Li, J., Deng, H., Yang, J. & Wu, Z.  
    **Is the Kernel–Staples Match a Key–Lock Match?**  
    *Chemical Science* **9**, 2437–2442 (2018).  
    https://doi.org/10.1039/C7SC05019D

17. Zhuang, S., Liao, L., Yuan, J., Xia, N., Zhao, Y., Wang, C., Gan, Z., Yan, N., He, L., Li, J., Deng, H., Guan, Z., Yang, J. & Wu, Z.  
    **Fcc versus Non-fcc Structural Isomerism of Gold Nanoparticles with Kernel Atom Packing Dependent Photoluminescence.**  
    *Angewandte Chemie International Edition* **58**, 4510–4514 (2019).  
    https://doi.org/10.1002/anie.201813426

18. Zeng, C., Chen, Y., Iida, K., Nobusada, K., Kirschbaum, K., Lambright, K. J. & Jin, R.  
    **Gold Quantum Boxes: On the Periodicities and the Quantum Confinement in the Au28, Au36, Au44, and Au52 Magic Series.**  
    *Journal of the American Chemical Society* **138**, 3950–3953 (2016).  
    https://doi.org/10.1021/jacs.5b12747

19. Zhuang, S., Liao, L., Li, M.-B., Yao, C., Zhao, Y., Dong, H., Li, J., Deng, H., Li, L. & Wu, Z.  
    **The fcc Structure Isomerization in Gold Nanoclusters.**  
    *Nanoscale* **9**, 14809–14813 (2017).  
    https://doi.org/10.1039/C7NR05239A

20. Zeng, C., Liu, C., Chen, Y., Rosi, N. L. & Jin, R.  
    **Atomic Structure of Self-Assembled Monolayer of Thiolates on a Tetragonal Au92 Nanocrystal.**  
    *Journal of the American Chemical Society* **138**, 8710–8713 (2016).  
    https://doi.org/10.1021/jacs.6b04835

21. Heinecke, C. L., Ni, T. W., Malola, S., Mäkinen, V., Wong, O. A., Häkkinen, H. & Ackerson, C. J.  
    **Structural and Theoretical Basis for Ligand Exchange on Thiolate Monolayer Protected Gold Nanoclusters.**  
    *Journal of the American Chemical Society* **134**, 13316–13322 (2012).  
    https://doi.org/10.1021/ja3032339

22. Jadzinsky, P. D., Calero, G., Ackerson, C. J., Bushnell, D. A. & Kornberg, R. D.  
    **Structure of a Thiol Monolayer-Protected Gold Nanoparticle at 1.1 Å Resolution.**  
    *Science* **318**, 430–433 (2007).  
    https://doi.org/10.1126/science.1148624

23. Yan, N., Xia, N., Liao, L., Zhu, M., Jin, F., Jin, R. & Wu, Z.  
    **Unraveling the Long-Pursued Au144 Structure by X-ray Crystallography.**  
    *Science Advances* **4**, eaat7259 (2018).  
    https://doi.org/10.1126/sciadv.aat7259

24. Sakthivel, N. A., Theivendran, S., Ganeshraj, V., Oliver, A. G. & Dass, A.  
    **Crystal Structure of Faradaurate-279: Au279(SPh-tBu)84 Plasmonic Nanocrystal Molecules.**  
    *Journal of the American Chemical Society* **139**, 15450–15459 (2017).  
    https://doi.org/10.1021/jacs.7b08651

25. Chen, C. & Ong, S. P.  
    **A Universal Graph Deep Learning Interatomic Potential for the Periodic Table.**  
    *Nature Computational Science* **2**(11), 718–728 (2022).  
    https://doi.org/10.1038/s43588-022-00349-3

26. Perego, S. & Bonati, L.  
    **Data Efficient Machine Learning Potentials for Modeling Catalytic Reactivity via Active Learning and Enhanced Sampling.**  
    *npj Computational Materials* **10**, 291 (2024).  
    https://doi.org/10.1038/s41524-024-01481-6

27. Materials Project.  
    **Matbench Discovery.**  
    https://matbench-discovery.materialsproject.org/  
    Accessed: 10 July 2026.

28. MACE Developers.  
    **MACE Documentation.**  
    https://mace-docs.readthedocs.io/en/latest/quick-start/quick-start.html   
    Accessed: 1 October 2026.